In [ ]:
#| default_exp execute

## Executing notebooks

Execute notebooks with visible outputs, local package imports, and a fast per-cell timeout. Cells that exceed the timeout are marked with execution-fingerprint metadata and skipped on later runs until their source changes.

Execution closes the loop after reading and writing. The project needs a way to run notebooks as notebooks, with local imports available and with visible outputs copied back into the notebook for inspection.

This notebook wraps `execnb` with nbdev-friendly defaults: local import paths, optional partial execution, timeout handling, and a test helper that reports notebook errors in a concise form.

Execution is the confidence step after an edit. The wrapper keeps imports close to how nbdev users run notebooks, records visible outputs for inspection, and marks timed-out cells so repeated runs do not keep blocking on the same long operation.

```python
exec_nb("nbs/03_execute.ipynb", up2id="some-cell-id", timeout=10, show_output=True)
```

### Production contract

Notebook execution is production core. `exec_nb` must run notebooks with local project imports available, support partial execution by cell or chapter, capture visible output, keep safe mode on by default, mark timed-out cells so repeated runs do not trap users, and recover when timed-out source changes.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from fastcore.nbio import read_nb
from fastcore.test import test_eq
from nbskill.execute import exec_nb as _example_exec_nb
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook, write_tool_notebook
from nbskill.write import update_cell, write_nb

In [ ]:
with write_demo_notebook("03_execute_example.ipynb") as path:
    _example_write_nb(str(path), "%%code\nvalue = 6 * 7\nprint(value)", replace=True)
    _example_exec_nb(str(path), timeout=5, show_output=True)

In [ ]:
#| export
import ast
import asyncio
import base64
import hashlib
import json
import re
import site
import sys
import threading
import traceback
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path
from typing import Annotated

import pyskills
from execnb.shell import CaptureShell
from fastcore.nbio import read_nb
from fastcore.nbio import write_nb as _write_nb

from nbskill.foundation import (
    nbskill_cell_metadata, cell_source, cli_error, cli_return, one_chapter, output_text,
    output_value_text, parse_literal, source_hash, stamp_notebook_metadata,
)
from nbskill.parallel import execution_slot, notebook_locks

Fastcore 2 removes `fastcore.script.Param`, and current `safepyrun` still imports it at module import time. Keep `safepyrun` lazy so MCP startup and non-execution tools can initialize, then install a tiny `Param` compatibility alias only when the safe runner is actually needed.

In [ ]:
#| export
def _ensure_safepyrun_fastcore_param():
    import fastcore.script as fastcore_script
    if hasattr(fastcore_script, "Param"): return

    def Param(desc="", typ=str, **kwargs): return Annotated[typ, desc, kwargs]
    fastcore_script.Param = Param

In [ ]:
#| export
def _safepyrun_imports():
    _ensure_safepyrun_fastcore_param()
    from safepyrun import RunPython
    from safepyrun.core import allow
    return RunPython, allow

In [ ]:
#| export
def _safepyrun_allow(*args, **kwargs):
    _, allow = _safepyrun_imports()
    return allow(*args, **kwargs)

### Choosing how much to run

Sometimes verification only needs the first few cells or a single chapter. These helpers translate an index, a cell id, or a chapter into pre/post hooks that stop execution at the right point.

In [ ]:
#| export
def _exec_limiters(up2id):
    up2id = parse_literal(up2id)
    noop = lambda cell: None
    if up2id is None: return (lambda cell: False), noop
    if isinstance(up2id, int):
        if up2id < 0: raise ValueError("up2id must be >= 0")
        return (lambda cell: cell.idx_ >= up2id), noop

    done = False
    def preproc(cell): return done
    def postproc(cell):
        nonlocal done
        if cell.id == str(up2id): done = True
    return preproc, postproc

### Importing like the project does

Notebook execution should see the same local package that tests and examples see. This section finds the project root from common markers and puts the notebook folder, root, and optional `src` folder on the shell path.

In [ ]:
#| export
def _project_root_for_notebook(path):
    path = Path(path).resolve()
    start = path.parent if path.suffix else path
    markers = ("pyproject.toml", "settings.ini", "nbdev.yml", ".git")
    for folder in (start, *start.parents):
        if any((folder / marker).exists() for marker in markers): return folder
    if start.name in {"nbs", "notebooks"} and start.parent != start.parent.parent: return start.parent
    return start

In [ ]:
#| export
def _local_import_paths(path):
    nb_dir = Path(path).resolve().parent
    root = _project_root_for_notebook(path)
    paths = [nb_dir, root]
    src = root / "src"
    if src.exists(): paths.append(src)
    return [p for i, p in enumerate(paths) if p.exists() and p not in paths[:i]]

In [ ]:
#| export
_SAFE_SENTINEL = "__nbskill_safe_exec__"
_EXECUTION_POLICY_FILE = ".nbskill-exec-approval.json"
_EXECUTION_POLICY_VERSION = 1

Safe execution also keeps a project policy file named `.nbskill-exec-approval.json`. Each run syncs top-level function names from the notebook into that file. Functions with no statically detected external effects default to `allowed: true`; functions that look like they can write files, spawn processes, remove files, or make network calls default to `allowed: false` until the project file is edited.

In [ ]:
#| export
def _execution_policy_path(path):
    return _project_root_for_notebook(path) / _EXECUTION_POLICY_FILE

In [ ]:
#| export
def _empty_execution_policy():
    return {"version": _EXECUTION_POLICY_VERSION, "functions": {}}

In [ ]:
#| export
def _read_execution_policy(path):
    policy_path = _execution_policy_path(path)
    if policy_path.exists():
        try:
            policy = json.loads(policy_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid nbskill execution policy {policy_path}: {exc}") from exc
        if not isinstance(policy, dict): raise ValueError(f"Invalid nbskill execution policy {policy_path}: expected object")
    else:
        policy = _empty_execution_policy()
    policy.setdefault("version", _EXECUTION_POLICY_VERSION)
    if not isinstance(policy.setdefault("functions", {}), dict):
        raise ValueError(f"Invalid nbskill execution policy {policy_path}: functions must be an object")
    policy["_path"] = str(policy_path)
    return policy

In [ ]:
#| export
def _write_execution_policy(policy):
    policy_path = policy.get("_path")
    if not policy_path: raise ValueError("execution policy is missing _path")
    payload = {key: value for key, value in policy.items() if not str(key).startswith("_")}
    Path(policy_path).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

In [ ]:
#| export
_EXTERNAL_EFFECT_CALLS = {
    "aiosqlite.connect", "asyncpg.connect", "httpx.delete", "httpx.get", "httpx.patch", "httpx.post", "httpx.put", "httpx.request", "httpx.stream",
    "psycopg.connect", "psycopg2.connect", "pymongo.MongoClient", "redis.Redis", "redis.StrictRedis",
    "os.chmod", "os.chown", "os.makedirs", "os.mkdir", "os.popen", "os.remove", "os.replace", "os.rename",
    "os.rmdir", "os.symlink", "os.system", "os.unlink",
    "requests.delete", "requests.get", "requests.patch", "requests.post", "requests.put", "requests.request",
    "shutil.copy", "shutil.copy2", "shutil.copytree", "shutil.move", "shutil.rmtree",
    "socket.create_connection", "socket.socket", "sqlalchemy.create_async_engine", "sqlalchemy.create_engine", "sqlite3.connect",
    "subprocess.Popen", "subprocess.call", "subprocess.check_call", "subprocess.check_output", "subprocess.run",
    "urllib.request.urlopen",
}
_EXTERNAL_EFFECT_ATTRS = {
    "chmod", "hardlink_to", "mkdir", "rename", "replace", "rmdir", "symlink_to", "touch", "unlink",
    "write", "write_bytes", "write_text", "writelines",
}
_SAFE_POLICY_CALLS = {
    "abs", "all", "any", "bool", "bytes", "dict", "enumerate", "float", "getattr", "hasattr", "int", "isinstance",
    "issubclass", "len", "list", "max", "min", "print", "range", "repr", "set", "sorted", "str", "sum", "tuple", "zip",
}

In [ ]:
#| export
def _execution_import_aliases_from_tree(tree):
    aliases = {}
    for node in tree.body:
        if isinstance(node, ast.Import):
            for item in node.names:
                aliases[item.asname or item.name.split(".")[0]] = item.name
        elif isinstance(node, ast.ImportFrom) and node.module:
            for item in node.names:
                if item.name == "*": continue
                aliases[item.asname or item.name] = f"{node.module}.{item.name}"
    return aliases

In [ ]:
#| export
def _execution_policy_import_aliases(nb):
    aliases = {}
    for cell in nb.cells:
        if cell.cell_type != "code": continue
        try: tree = ast.parse(cell_source(cell))
        except SyntaxError: continue
        aliases.update(_execution_import_aliases_from_tree(tree))
    return aliases

In [ ]:
#| export
def _call_name(node, aliases=None):
    aliases = aliases or {}
    if isinstance(node, ast.Name): return aliases.get(node.id, node.id)
    if isinstance(node, ast.Attribute):
        base = _call_name(node.value, aliases=aliases)
        return f"{base}.{node.attr}" if base else node.attr
    return ""

In [ ]:
#| export
def _open_write_mode(call):
    mode = "r"
    if len(call.args) > 1 and isinstance(call.args[1], ast.Constant) and isinstance(call.args[1].value, str):
        mode = call.args[1].value
    for keyword in call.keywords:
        if keyword.arg == "mode" and isinstance(keyword.value, ast.Constant) and isinstance(keyword.value.value, str):
            mode = keyword.value.value
    return any(char in mode for char in "wax+")

In [ ]:
#| export
def _external_effect_reasons(call, aliases=None):
    name = _call_name(call.func, aliases=aliases)
    attr = call.func.attr if isinstance(call.func, ast.Attribute) else ""
    reasons = []
    if (name == "open" or name.endswith(".open")) and _open_write_mode(call):
        reasons.append("call:open(write)")
    if name in _EXTERNAL_EFFECT_CALLS:
        reasons.append(f"call:{name}")
    if attr in _EXTERNAL_EFFECT_ATTRS:
        reasons.append(f"call:{name or attr}")
    return reasons

In [ ]:
#| export
def _node_external_effect_reasons(node, aliases=None):
    reasons = []
    for child in ast.walk(node):
        if isinstance(child, ast.Call): reasons.extend(_external_effect_reasons(child, aliases=aliases))
    return sorted(set(reasons))

In [ ]:
#| export
def _function_runtime_nodes(node):
    nodes = [*node.decorator_list, *node.args.defaults, *(value for value in node.args.kw_defaults if value)]
    if node.returns: nodes.append(node.returns)
    args = [*node.args.posonlyargs, *node.args.args, *node.args.kwonlyargs]
    if node.args.vararg: args.append(node.args.vararg)
    if node.args.kwarg: args.append(node.args.kwarg)
    nodes += [arg.annotation for arg in args if arg.annotation]
    return nodes

In [ ]:
#| export
def _function_runtime_external_effect_reasons(node, aliases=None):
    reasons = []
    for part in _function_runtime_nodes(node):
        reasons.extend(_node_external_effect_reasons(part, aliases=aliases))
    return sorted(set(reasons))

In [ ]:
#| export
def _execution_policy_function_defs(nb):
    functions = {}
    for cell in nb.cells:
        if cell.cell_type != "code": continue
        try: tree = ast.parse(cell_source(cell))
        except SyntaxError: continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name not in functions:
                functions[node.name] = {"node": node, "cell_id": getattr(cell, "id", ""), "line": node.lineno, "source_hash": source_hash(cell_source(cell))}
    return functions

In [ ]:
#| export
def _raw_call_names(node):
    return {child.func.id for child in ast.walk(node) if isinstance(child, ast.Call) and isinstance(child.func, ast.Name)}

In [ ]:
#| export
def _execution_policy_function_effects(functions, aliases=None):
    aliases = {key: value for key, value in (aliases or {}).items() if key not in functions}
    effects = {name: set(_node_external_effect_reasons(info["node"], aliases=aliases)) for name, info in functions.items()}
    local_calls = {name: _raw_call_names(info["node"]) & set(functions) for name, info in functions.items()}
    changed = True
    while changed:
        changed = False
        for name, callees in local_calls.items():
            for callee in callees:
                for reason in effects[callee]:
                    indirect = f"{callee}->{reason}"
                    if indirect not in effects[name]:
                        effects[name].add(indirect)
                        changed = True
    return {name: sorted(reasons) for name, reasons in effects.items()}

In [ ]:
#| export
def _execution_policy_notebook(path):
    root = _project_root_for_notebook(path).resolve()
    full = Path(path).resolve()
    try: return str(full.relative_to(root))
    except ValueError: return str(full)

In [ ]:
#| export
def _coerce_execution_policy_entry(entry, default_allowed, source_hash=None):
    if isinstance(entry, dict):
        entry = dict(entry)
        allowed = entry.get("allowed", default_allowed)
    else:
        allowed = entry if isinstance(entry, bool) else default_allowed
        entry = {}
    if source_hash and entry.get("source_hash") != source_hash: allowed = default_allowed
    entry["allowed"] = bool(allowed)
    return entry

### Keeping project approvals current

`_sync_execution_policy` removes an approval only when the entry’s recorded cell no longer exists in its notebook. This keeps deleted cells from accumulating without removing approvals owned by other notebooks.

In [ ]:
#| exporti
def _prune_execution_policy_entries(path, nb, entries, notebook):
    root = _project_root_for_notebook(path)
    cell_ids = {notebook: {str(cell.id) for cell in nb.cells}}
    for name, entry in list(entries.items()):
        if not isinstance(entry, dict) or not entry.get("cell_id") or not entry.get("notebook"): continue
        entry_notebook = entry["notebook"]
        if entry_notebook not in cell_ids:
            notebook_path = Path(entry_notebook)
            if not notebook_path.is_absolute(): notebook_path = root / notebook_path
            try: cell_ids[entry_notebook] = {str(cell.id) for cell in read_nb(notebook_path).cells}
            except OSError: cell_ids[entry_notebook] = set()
        if str(entry["cell_id"]) not in cell_ids[entry_notebook]: entries.pop(name)

In [ ]:
#| export
def _sync_execution_policy(path, nb):
    policy = _read_execution_policy(path)
    functions = _execution_policy_function_defs(nb)
    aliases = _execution_policy_import_aliases(nb)
    effects = _execution_policy_function_effects(functions, aliases=aliases)
    entries = policy.setdefault("functions", {})
    notebook = _execution_policy_notebook(path)
    _prune_execution_policy_entries(path, nb, entries, notebook)

    for name in sorted(functions):
        reasons = effects.get(name, [])
        entry = _coerce_execution_policy_entry(entries.get(name), not bool(reasons), functions[name]["source_hash"])
        entry.update({
            "cell_id": str(functions[name]["cell_id"]),
            "source_hash": functions[name]["source_hash"],
            "external_effects": bool(reasons),
            "line": int(functions[name]["line"]),
            "notebook": notebook,
            "reasons": reasons,
        })
        entries[name] = entry
    _write_execution_policy(policy)
    return policy

In [ ]:
#| export
def _cell_policy_tree(cell):
    if cell.cell_type != "code": return None
    try: return ast.parse(cell_source(cell))
    except SyntaxError: return None

In [ ]:
#| export
def _cell_policy_function_names(cell):
    tree = _cell_policy_tree(cell)
    if tree is None: return []
    return [node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]

In [ ]:
#| export
def _top_level_external_effect_reasons(tree, aliases=None):
    reasons = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            reasons.extend(_function_runtime_external_effect_reasons(node, aliases=aliases))
        elif isinstance(node, ast.ClassDef):
            reasons.append("class-body")
        else:
            reasons.extend(_node_external_effect_reasons(node, aliases=aliases))
    return sorted(set(reasons))

In [ ]:
#| export
def _top_level_policy_call_names(tree, aliases=None):
    names = set()
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom)):
            continue
        for child in ast.walk(node):
            if isinstance(child, ast.Call) and (name := _call_name(child.func, aliases=aliases)):
                names.add(name)
    return names

In [ ]:
#| export
def _policy_entry_allowed(policy, name):
    if not isinstance(policy, dict): return False
    entry = policy.get("functions", {}).get(name)
    if isinstance(entry, dict): return entry.get("allowed") is True
    return entry is True

In [ ]:
#| export
def _policy_allowed_function_names(policy):
    if not isinstance(policy, dict): return []
    return [name for name in policy.get("functions", {}) if _policy_entry_allowed(policy, name)]

In [ ]:
#| export
def _cell_has_project_execution_approval(cell, policy=None):
    if not policy: return False
    tree = _cell_policy_tree(cell)
    if tree is None: return False
    aliases = _execution_import_aliases_from_tree(tree)
    if any(isinstance(node, ast.ClassDef) for node in tree.body): return False
    defined = [node for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
    if any(node.decorator_list for node in defined): return False
    if _top_level_external_effect_reasons(tree, aliases=aliases): return False
    calls = _top_level_policy_call_names(tree, aliases=aliases)
    policy_functions = set(policy.get("functions", {}))
    policy_calls = calls & policy_functions
    unknown = calls - policy_calls - _SAFE_POLICY_CALLS
    if unknown: return False
    needed = {node.name for node in defined} | policy_calls
    return bool(needed) and all(_policy_entry_allowed(policy, name) for name in needed)

In [ ]:
#| export
def _policy_blocked_function_names(cell, policy=None):
    if not policy: return []
    tree = _cell_policy_tree(cell)
    if tree is None: return []
    aliases = _execution_import_aliases_from_tree(tree)
    policy_functions = set(policy.get("functions", {}))
    names = [node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
    names += [name for name in _top_level_policy_call_names(tree, aliases=aliases) if name in policy_functions]
    return sorted({name for name in names if not _policy_entry_allowed(policy, name)})

In [ ]:
#| export
def _register_project_allowed(policy, namespace):
    for name in _policy_allowed_function_names(policy):
        if name in namespace: _safepyrun_allow(namespace[name])

In [ ]:
#| export
_DEFAULT_CACHE_DOMAINS = (
    "chatgpt.com", "api.openai.com", "api.anthropic.com",
    "generativelanguage.googleapis.com", "api.deepseek.com",
    "api.fireworks.ai", "openrouter.ai", "api.groq.com",
    "api.together.xyz", "api.mistral.ai", "api.x.ai", "api.moonshot.ai",
)


In [ ]:
#| export
_CACHY_NORM_PATS = [
    (re.compile(r"/ipykernel_\d+/\d+\.py"), "/ipykernel_N/X.py"),
    (re.compile(r"<ipython-input-\d+-\w+>"), "<ipython-input>"),
    (re.compile(r"/var/folders/[\w|/]+"), "/tmp/tmpT"),
    (re.compile(r"/tmp/tmp\w+"), "/tmp/tmpX"),
    (re.compile(r"ipykernel_\d+"), "ipykernel_N"),
    (re.compile(r"0x[0-9a-fA-F]{6,}"), "0xMEM"),
    (re.compile(r"line \d+, in"), "line N, in"),
]


In [ ]:
#| export
def _parse_str_list(value, default=None):
    value = parse_literal(value)
    if value is None: return list(default or [])
    if isinstance(value, (list, tuple, set)): return [str(o) for o in value if str(o).strip()]
    if isinstance(value, str): return [part.strip() for part in value.split(",") if part.strip()]
    return [str(value)]


In [ ]:
#| export
def _project_env_site_paths(path):
    root = _project_root_for_notebook(path)
    candidates = []
    for name in (".venv", "venv"):
        env = root / name
        candidates.append(env / "Lib" / "site-packages")
        lib = env / "lib"
        current = lib / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
        if current.exists(): candidates.append(current)
        elif lib.exists(): candidates.extend(sorted(lib.glob("python*/site-packages")))
    return [path for i, path in enumerate(candidates) if path.exists() and path not in candidates[:i]]


In [ ]:
#| export
def _prepend_sys_path(path):
    path = str(Path(path))
    if path in sys.path: sys.path.remove(path)
    sys.path.insert(0, path)


In [ ]:
#| export
@contextmanager
def _temporary_sys_path(paths):
    old_path = list(sys.path)
    for path in reversed([Path(p) for p in paths]):
        if path.name == "site-packages":
            before = list(sys.path)
            site.addsitedir(str(path))
            for added in reversed([item for item in sys.path if item not in before]):
                _prepend_sys_path(added)
        else:
            _prepend_sys_path(path)
    try: yield
    finally: sys.path[:] = old_path


In [ ]:
#| export
@contextmanager
def _temporary_allow_registry():
    snapshot = {key: set(value) for key, value in pyskills.__pytools__.items()}
    try: yield
    finally:
        pyskills.__pytools__.clear()
        for key, value in snapshot.items(): pyskills.__pytools__[key].update(value)


In [ ]:
#| export
def _resolve_allowed_name(name, namespace):
    if name in namespace: return namespace[name]
    parts = str(name).split(".")
    for i in range(len(parts), 0, -1):
        module_name = ".".join(parts[:i])
        try:
            module = __import__(module_name, fromlist=["*"])
            obj = module
            for part in parts[i:]: obj = getattr(obj, part)
            return obj
        except (ImportError, AttributeError):
            continue
    raise ValueError(f"Could not resolve allow entry {name!r}")


In [ ]:
#| export
def _register_allowed(allow, namespace):
    for name in _parse_str_list(allow):
        _safepyrun_allow(_resolve_allowed_name(name, namespace))


In [ ]:
#| export
def _cachy_content(request):
    if not hasattr(request, "_content"): request.read()
    content_type = request.headers.get("Content-Type", "").encode()
    boundary = None
    try:
        import httpx
        boundary = httpx._multipart.get_multipart_boundary_from_content_type(content_type)
    except Exception:
        boundary = None
    return request.content.replace(boundary, b"cachy-boundary") if boundary else request.content


In [ ]:
#| export
def _cachy_normalize(data):
    text = data.decode("utf-8", errors="replace")
    for pattern, replacement in _CACHY_NORM_PATS: text = pattern.sub(replacement, text)
    return text.encode("utf-8")


In [ ]:
#| export
def _cachy_norm_content(request):
    content = _cachy_content(request)
    if "json" in request.headers.get("content-type", "").lower():
        try: return json.dumps(json.loads(content), sort_keys=True).encode()
        except Exception: pass
    return content


In [ ]:
#| export
def _cachy_key(request, is_stream=False):
    url = request.url.copy_remove_param("key")
    data = f"{url}{bool(is_stream)}".encode() + _cachy_normalize(_cachy_norm_content(request))
    return hashlib.sha256(data).hexdigest()[:8]


In [ ]:
#| export
def _cache_path_for_notebook(path, cache_dir=None):
    base = Path(cache_dir) if cache_dir else _project_root_for_notebook(path)
    return base / "cachy.jsonl"


In [ ]:
#| export
def _cached_response(key, cache_path, request):
    if not cache_path.exists(): return None
    import httpx
    with cache_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip(): continue
            entry = json.loads(line)
            if entry.get("key") != key: continue
            content = entry.get("response", "")
            if entry.get("binary"): content = base64.b64decode(content)
            return httpx.Response(
                status_code=entry.get("status_code", 200),
                content=content,
                headers=entry.get("headers"),
                request=request,
            )
    return None


In [ ]:
#| export
def _url_allowed(url, domains):
    return any(domain in str(url) for domain in domains)


In [ ]:
#| export
@contextmanager
def _httpx_guard(path, cache_httpx=False, cache_dir=None, cache_domains=None):
    try: import httpx
    except ImportError:
        yield
        return
    _safepyrun_allow({
        httpx: ["get", "post", "put", "patch", "delete", "head", "options", "request", "stream"],
        httpx.Client: ["send", "request", "get", "post", "put", "patch", "delete", "head", "options", "stream"],
        httpx.AsyncClient: ["send", "request", "get", "post", "put", "patch", "delete", "head", "options", "stream"],
    })
    original_sync, original_async = httpx.Client.send, httpx.AsyncClient.send
    original_funcs = {
        name: getattr(httpx, name)
        for name in ("request", "get", "post", "put", "patch", "delete", "head", "options")
        if hasattr(httpx, name)
    }
    domains = tuple(_parse_str_list(cache_domains, default=_DEFAULT_CACHE_DOMAINS))
    cache_path = _cache_path_for_notebook(path, cache_dir)

    def from_cache(request, is_stream):
        if not cache_httpx:
            raise RuntimeError(f"nbskill safe execution blocked live httpx call to {request.url}")
        if not _url_allowed(request.url, domains):
            raise RuntimeError(f"nbskill safe execution has no cached domain rule for {request.url}")
        key = _cachy_key(request, is_stream=is_stream)
        response = _cached_response(key, cache_path, request)
        if response is None:
            raise RuntimeError(f"nbskill safe execution has no cached httpx response for {request.url} (key={key})")
        return response

    def send(self, request, **kwargs):
        return from_cache(request, kwargs.get("stream", False))

    async def asend(self, request, **kwargs):
        return from_cache(request, kwargs.get("stream", False))

    def request(method, url, **kwargs):
        req = httpx.Request(
            method, url, params=kwargs.get("params"), headers=kwargs.get("headers"),
            content=kwargs.get("content"), data=kwargs.get("data"), json=kwargs.get("json"),
        )
        return from_cache(req, kwargs.get("stream", False))

    def method_request(method):
        return lambda url, **kwargs: request(method, url, **kwargs)

    httpx.Client.send = send
    httpx.AsyncClient.send = asend
    httpx.request = request
    for method in ("get", "post", "put", "patch", "delete", "head", "options"):
        setattr(httpx, method, method_request(method.upper()))
    try: yield
    finally:
        httpx.Client.send = original_sync
        httpx.AsyncClient.send = original_async
        for name, func in original_funcs.items(): setattr(httpx, name, func)


In [ ]:
#| export
def _run_async(coro):
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(coro)
    box = {}
    def target():
        try: box["result"] = asyncio.run(coro)
        except BaseException as exc: box["exc"] = exc
    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    if "exc" in box: raise box["exc"]
    return box.get("result")


In [ ]:
#| export
def _stream_output(name, text):
    if not text: return None
    return {"output_type": "stream", "name": name, "text": text}


In [ ]:
#| export
def _result_output(value):
    return {
        "output_type": "execute_result",
        "execution_count": None,
        "metadata": {},
        "data": {"text/plain": repr(value)},
    }


In [ ]:
#| export
def _error_output(exc):
    tb = traceback.format_exception(type(exc), exc, exc.__traceback__)
    return {"output_type": "error", "ename": type(exc).__name__, "evalue": str(exc), "traceback": tb}


In [ ]:
#| export
def _magic_error(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith("!") or stripped.startswith("%"):
            return RuntimeError("nbskill safe execution blocks IPython shell escapes and magics")
    return None


In [ ]:
#| export
def _sync_safe_callable_globals(value, namespace):
    if hasattr(value, "__globals__"):
        value.__globals__.update(namespace)
    if isinstance(value, type):
        for item in vars(value).values():
            if hasattr(item, "__globals__"): item.__globals__.update(namespace)

In [ ]:
#| export
def _sync_safe_globals(namespace):
    for value in list(namespace.values()):
        _sync_safe_callable_globals(value, namespace)

In [ ]:
#| export
def _module_source(node):
    return ast.unparse(ast.Module(body=[node], type_ignores=[]))

In [ ]:
#| export
async def _run_safe_source(runner, namespace, source):
    tree = ast.parse(source)
    result = None
    for idx, node in enumerate(tree.body):
        last_expr = idx == len(tree.body) - 1 and isinstance(node, ast.Expr)
        chunk = ast.unparse(ast.Expression(node.value)) if last_expr else _module_source(node)
        result = await runner(chunk)
        _sync_safe_globals(namespace)
    return result

Safepyrun 0.2 defaults to blocking `def` statements before nbskill's project policy can approve them. We keep definitions enabled in the runner and let the notebook execution policy decide which functions are allowed to run.

In [ ]:
#| export
class _SafeShell:
    def __init__(
        self,
        path,
        extra_paths=None,
        allow=None,
        ok_dests=None,
        cache_httpx=False,
        cache_dir=None,
        cache_domains=None,
        execution_policy=None,
    ):
        self.path = Path(path)
        self.paths = [*(_local_import_paths(path)), *(_project_env_site_paths(path)), *(extra_paths or [])]
        self.cache_httpx = cache_httpx
        self.cache_dir = cache_dir
        self.cache_domains = cache_domains
        self.execution_policy = execution_policy
        self.safe = True
        self.exc = None
        self.g = {
            _SAFE_SENTINEL: True,
            "__name__": "__main__",
            "__file__": str(self.path),
        }
        RunPython, _ = _safepyrun_imports()
        with _temporary_sys_path(self.paths):
            _register_allowed(allow, self.g)
            _register_project_allowed(self.execution_policy, self.g)
        self.runner = RunPython(g=self.g, ok_dests=_parse_str_list(ok_dests), ban_defs=False)

    def run(self, source, timeout=30, verbose=False):
        self.exc = _magic_error(source)
        if self.exc: return [_error_output(self.exc)]
        out, err = StringIO(), StringIO()
        result = None
        try:
            async def call_runner():
                with _temporary_sys_path(self.paths), _httpx_guard(
                    self.path, cache_httpx=self.cache_httpx, cache_dir=self.cache_dir, cache_domains=self.cache_domains,
                ), redirect_stdout(out), redirect_stderr(err):
                    result = await _run_safe_source(self.runner, self.g, source)
                    _register_project_allowed(self.execution_policy, self.g)
                    return result
            coro = call_runner()
            if timeout and timeout > 0: coro = asyncio.wait_for(coro, timeout=timeout)
            result = _run_async(coro)
        except (asyncio.TimeoutError, TimeoutError):
            self.exc = TimeoutError(f"cell ran longer than {timeout}s")
        except BaseException as exc:
            self.exc = exc
        if verbose:
            if out.getvalue(): print(out.getvalue(), end="")
            if err.getvalue(): print(err.getvalue(), end="", file=sys.stderr)
        outputs = [o for o in (_stream_output("stdout", out.getvalue()), _stream_output("stderr", err.getvalue())) if o]
        if self.exc: outputs.append(_error_output(self.exc))
        elif result is not None: outputs.append(_result_output(result))
        return outputs

In [ ]:
#| export
def _exec_shell(
    path,
    extra_paths=None,
    safe=True,
    allow=None,
    ok_dests=None,
    cache_httpx=False,
    cache_dir=None,
    cache_domains=None,
    execution_policy=None,
):
    paths = [*(_local_import_paths(path)), *(_project_env_site_paths(path)), *(extra_paths or [])]
    if safe:
        return _SafeShell(
            path, extra_paths=extra_paths, allow=allow, ok_dests=ok_dests,
            cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains,
            execution_policy=execution_policy,
        )
    shell = CaptureShell()
    for pth in reversed(paths):
        shell.set_path(pth)
    return shell

### Timeouts that do not trap future runs

A timed-out cell can make repeated verification painful. Timeout metadata records the source fingerprint that timed out, skips that exact source on later runs, and automatically clears the mark when the cell changes.

In [ ]:
#| export
_TIMEOUT_HASH_KEY = "timeout_hash"

In [ ]:
#| export
_TIMEOUT_SECONDS_KEY = "timeout_seconds"

_EXECUTED_HASH_KEY = "executed_hash"

In [ ]:
#| export
# Execution approval and timeout markers share nbskill cell metadata with foundation.

In [ ]:
#| export
def _cell_source_hash(cell): return source_hash(cell.get("source", ""), length=None)

In [ ]:
#| export
class _ExecutionApprovalRequired(RuntimeError): pass


In [ ]:
#| export
def _cell_has_execution_approval(cell, execution_policy=None):
    if getattr(cell, "execution_count", None) not in (None, 0): return True
    info = nbskill_cell_metadata(cell, create=False) or {}
    if info.get(_EXECUTED_HASH_KEY) == _cell_source_hash(cell): return True
    return _cell_has_project_execution_approval(cell, execution_policy)

In [ ]:
#| export
def _approval_required_error(cell, execution_policy=None):
    policy_path = execution_policy.get("_path") if isinstance(execution_policy, dict) else None
    blocked = _policy_blocked_function_names(cell, execution_policy)
    if blocked and policy_path:
        policy_hint = f" Project policy {policy_path} has allowed=false for: {', '.join(blocked)}."
    elif policy_path:
        policy_hint = f" Project policy {policy_path} did not approve this cell."
    else:
        policy_hint = ""
    msg = (
        f"nbskill: refusing to execute unapproved cell id={cell.id}."
        f"{policy_hint} Edit the policy entry, run it once yourself, or rerun nbskill with allow_new=True if you approve this source."
    )
    return _ExecutionApprovalRequired(msg)

In [ ]:
#| export
def _mark_executed(cell):
    if cell.cell_type == "code": nbskill_cell_metadata(cell)[_EXECUTED_HASH_KEY] = _cell_source_hash(cell)

In [ ]:
#| export
def _timeout_stream(text):
    return {"output_type": "stream", "name": "stderr", "text": text if text.endswith("\n") else text + "\n"}

In [ ]:
#| export
def _timeout_error(ename, text):
    return {"output_type": "error", "ename": ename, "evalue": text, "traceback": [text]}

In [ ]:
#| export
def _cell_eval_false(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    for line in cell_source(cell).splitlines():
        text = line.strip().lower().replace(" ", "")
        if not text: continue
        if not text.startswith("#|"): return False
        if text in {"#|eval:false", "#|eval=false"}: return True
    return False

In [ ]:
#| export
def _skip_timed_out_cell(cell):
    if cell.cell_type != "code": return False
    meta = nbskill_cell_metadata(cell)
    current_hash = _cell_source_hash(cell)
    timeout_hash = meta.get(_TIMEOUT_HASH_KEY)
    if timeout_hash == current_hash:
        seconds = meta.get(_TIMEOUT_SECONDS_KEY, "?")
        msg = (
            f"nbskill: skipped cell id={cell.id}; it previously exceeded the "
            f"{seconds}s timeout. Edit the cell to change its hash and rerun it."
        )
        cell.outputs = [_timeout_error("NbskillTimeoutSkipped", msg)]
        return True
    if timeout_hash and timeout_hash != current_hash:
        meta.pop(_TIMEOUT_HASH_KEY, None)
        meta.pop(_TIMEOUT_SECONDS_KEY, None)
    return False

In [ ]:
#| export
def _mark_timeout(cell, timeout, outputs):
    meta = nbskill_cell_metadata(cell)
    meta[_TIMEOUT_HASH_KEY] = _cell_source_hash(cell)
    meta[_TIMEOUT_SECONDS_KEY] = timeout
    msg = f"nbskill: cell id={cell.id} ran longer than {timeout}s and was stopped."
    cell.outputs = [_timeout_stream(msg), *(outputs or [])]

In [ ]:
#| export
def _clear_timeout_mark(cell):
    meta = nbskill_cell_metadata(cell)
    meta.pop(_TIMEOUT_HASH_KEY, None)
    meta.pop(_TIMEOUT_SECONDS_KEY, None)

In [ ]:
#| export
def _run_cell(shell, cell, timeout=30, verbose=False, allow_new=False, execution_policy=None):
    if cell.cell_type != "code": return
    shell._cell_idx = cell.idx_ + 1
    if getattr(shell, "safe", False) and not allow_new and not _cell_has_execution_approval(cell, execution_policy):
        shell.exc = _approval_required_error(cell, execution_policy)
        cell.outputs = [_error_output(shell.exc)]
        return
    outputs = shell.run(cell.source, timeout=timeout if timeout and timeout > 0 else None, verbose=verbose)
    cell.outputs = outputs or []
    if isinstance(shell.exc, TimeoutError): _mark_timeout(cell, timeout, outputs)
    else:
        _clear_timeout_mark(cell)
        if shell.exc is None: _mark_executed(cell)

In [ ]:
#| export
def _execute_nb(
    path,
    dest=None,
    exc_stop=False,
    preproc=lambda cell: False,
    postproc=lambda cell: None,
    timeout=30,
    verbose=False,
    safe=True,
    allow=None,
    ok_dests=None,
    cache_httpx=False,
    cache_dir=None,
    cache_domains=None,
    allow_new=False,
):
    with notebook_locks(path, dest):
        with execution_slot():
            nb = read_nb(path)
            execution_policy = _sync_execution_policy(path, nb) if safe else None
            first_exc = None
            with _temporary_allow_registry():
                shell = _exec_shell(
                    path, safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx,
                    cache_dir=cache_dir, cache_domains=cache_domains, execution_policy=execution_policy,
                )
                for cell in nb.cells:
                    if preproc(cell): continue
                    if _cell_eval_false(cell):
                        cell.outputs = []
                        postproc(cell)
                        continue
                    if _skip_timed_out_cell(cell):
                        postproc(cell)
                        continue
                    _run_cell(
                        shell, cell, timeout=timeout, verbose=verbose, allow_new=allow_new,
                        execution_policy=execution_policy,
                    )
                    postproc(cell)
                    if isinstance(shell.exc, _ExecutionApprovalRequired):
                        break
                    if shell.exc and exc_stop:
                        first_exc = shell.exc
                        break
            if dest:
                stamp_notebook_metadata(nb)
                _write_nb(nb, dest)
            if first_exc: raise first_exc
            return nb

In [ ]:
#| export
def _is_rich_traceback_stream(output):
    if output.get("output_type") != "stream": return False
    text = output_value_text(output.get("text"))
    return "Traceback" in text and "\x1b[" in text


In [ ]:
#| export
def _executed_cells(nb, up2id=None):
    up2id = parse_literal(up2id)
    if up2id is None: return list(enumerate(nb.cells))
    if isinstance(up2id, int): return list(enumerate(nb.cells[:up2id]))
    items = []
    for idx, cell in enumerate(nb.cells):
        items.append((idx, cell))
        if cell.id == str(up2id): break
    return items


In [ ]:
#| export
def _print_outputs_from_nb(nb, up2id=None):
    for idx, cell in _executed_cells(nb, up2id):
        outputs = getattr(cell, "outputs", None) or []
        has_error = any(output.get("output_type") == "error" for output in outputs)
        for output in outputs:
            if has_error and _is_rich_traceback_stream(output): continue
            text = output_text(output)
            if not text: continue
            print(f"--- output id={cell.id} ---")
            print(text, end="" if text.endswith("\n") else "\n")


In [ ]:
#| export
def _print_nb_outputs(path, up2id=None):
    with notebook_locks(path):
        _print_outputs_from_nb(read_nb(path), up2id=up2id)


In [ ]:
#| export
def _parse_executed_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(getattr(cell, "source", ""))
    except SyntaxError: return None

In [ ]:
#| export
def _top_level_call_names(tree):
    calls = set()

    def visit(node):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): return
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name): calls.add(node.func.id)
        for child in ast.iter_child_nodes(node): visit(child)

    for node in tree.body: visit(node)
    return calls

In [ ]:
#| export
def _execution_warning_function(node):
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return False
    if node.decorator_list: return False
    return node.name == "main" or not node.name.startswith("_")

In [ ]:
#| export
def _execution_function_defs(nb, up2id=None):
    defs = {}
    for idx, cell in _executed_cells(nb, up2id=up2id):
        tree = _parse_executed_code_cell(cell)
        if tree is None: continue
        for node in tree.body:
            if _execution_warning_function(node):
                defs.setdefault(node.name, {"cell_id": getattr(cell, "id", ""), "line": getattr(node, "lineno", None)})
    return defs

In [ ]:
#| export
def _uncalled_function_warnings(nb, up2id=None):
    called = set()
    for idx, cell in _executed_cells(nb, up2id=up2id):
        tree = _parse_executed_code_cell(cell)
        if tree is not None: called.update(_top_level_call_names(tree))
    warnings = []
    for name, info in sorted(_execution_function_defs(nb, up2id=up2id).items()):
        if name in called: continue
        location = f"cell id={info['cell_id']}" if info.get("cell_id") else "an executed cell"
        warnings.append(
            f"function {name!r} defined in {location} was not called by executed cells; "
            "add a focused test or example, or add a decorator if it is invoked externally."
        )
    return warnings

In [ ]:
#| export
def _print_uncalled_function_warnings(nb, up2id=None):
    warnings = _uncalled_function_warnings(nb, up2id=up2id)
    if not warnings: return []
    print("Execution warnings:")
    for warning in warnings: print(f"- {warning}")
    return warnings

### The public executor

`exec_nb` is the user-facing wrapper around the execution engine. It writes outputs back to the notebook, prints visible outputs when requested, and supports partial execution through `up2id` or `chapter`.

In [ ]:
#| export
def _safe_mode_audit_blocked(nb, up2id=None):
    for cell in _executed_cells(nb, up2id=up2id):
        for output in getattr(cell, "outputs", ()):
            if "PermissionError: Audit:" in output_text(output): return True
    return False

In [ ]:
#| export
def _safe_mode_hint(nb, up2id=None):
    if not _safe_mode_audit_blocked(nb, up2id=up2id): return ""
    return "Safe execution hint: safepyrun blocked an audited operation. For trusted notebooks, rerun with safe=False or --no-safe."

In [ ]:
#| export
def exec_nb(
    path: str,  # Notebook path
    dest: str | None = None,  # Destination path; defaults to overwriting path
    exc_stop: bool = False,  # Stop on exceptions
    up2id: int | str | None = None,  # Execute first N cells, or through this cell id
    chapter: str | None = None,  # Execute through this chapter, inclusive
    timeout: int = 30,  # Per-cell timeout in seconds; <=0 disables timeouts
    show_output: bool = True,  # Print saved cell outputs and errors after execution
    verbose: bool = False,  # Show stdout/stderr live while executing
    safe: bool = True,  # Use safepyrun instead of the legacy execnb shell
    allow: str | None = None,  # Comma-separated or literal list of trusted callables to allow
    ok_dests: str | None = None,  # Comma-separated or literal list of allowed write destinations
    cache_httpx: bool = False,  # Return cached httpx responses instead of making live calls
    cache_dir: str | None = None,  # Directory containing cachy.jsonl; defaults to project root
    cache_domains: str | None = None,  # Comma-separated or literal list of cacheable domains
    allow_new: bool = False,  # Execute cells without prior user/nbskill execution approval
    check_only: bool = False,  # Run in memory without writing notebook outputs or metadata
):
    "Execute a notebook with safe Python by default and local project imports available."
    dest = None if check_only else (dest or path)
    chapter_title = None
    if chapter is not None:
        if up2id is not None: raise ValueError("Use either chapter or up2id, not both")
        with notebook_locks(path):
            nb = read_nb(path)
            span = one_chapter(nb.cells, chapter)
        up2id, chapter_title = span["end"], span["title"]
    preproc, postproc = _exec_limiters(up2id)
    nb = _execute_nb(
        path, dest=dest, exc_stop=exc_stop, preproc=preproc, postproc=postproc,
        timeout=timeout, verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests,
        cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains,
        allow_new=allow_new,
    )
    mode = "safe" if safe else "unsafe"
    target = "not written" if check_only else dest
    msg = f"Executed {path} -> {target} ({mode})"
    if check_only: msg += " (check_only=True)"
    if chapter_title is not None: msg += f" (chapter={chapter_title!r}, up2id={up2id})"
    elif up2id is not None: msg += f" (up2id={up2id})"
    if timeout and timeout > 0: msg += f" (timeout={timeout}s)"
    print(msg)
    if show_output:
        if check_only: _print_outputs_from_nb(nb, up2id=up2id)
        else: _print_nb_outputs(dest, up2id=up2id)
    if safe and (hint := _safe_mode_hint(nb, up2id=up2id)): print(hint)
    _print_uncalled_function_warnings(nb, up2id=up2id)
    return cli_return(Path(path) if check_only else Path(dest))

With fastcore 2, nbdev tests are treated as direct Python calls instead of CLI calls, so `exec_nb(check_only=True)` returns the notebook path while still leaving the file unchanged.

In [ ]:
#| hide
with write_tool_notebook("03_execute_tool.ipynb") as path:
    before = path.read_text(encoding="utf-8")
    result = exec_nb(str(path), timeout=5, allow_new=True, check_only=True, show_output=False)
    test_eq(result, path)
    test_eq(path.read_text(encoding="utf-8"), before)

In [ ]:
root = demo_path("03_execute_project")


In [ ]:
def _make_execute_project(name):
    root = demo_path(name)
    root.mkdir()
    (root / "pyproject.toml").write_text("[project]\nname = 'local-demo'\n", encoding="utf-8")
    pkg = root / "local_demo"
    pkg.mkdir()
    (pkg / "__init__.py").write_text("def meaning():\n    return 42\n", encoding="utf-8")
    nbs = root / "nbs"
    nbs.mkdir()
    return root, nbs


In [ ]:
root, nbs = _make_execute_project("03_execute_project")
try:
    path = nbs / "sample.ipynb"
    write_nb(str(path), "%%code\nfrom local_demo import meaning\nprint(meaning())\nassert meaning() == 42", replace=True)
    exec_nb(str(path), timeout=5, allow="local_demo.meaning")
    text = "".join(output_text(output) for output in read_nb(path).cells[0].outputs)
    assert "refusing to execute unapproved cell" in text
    exec_nb(str(path), timeout=5, allow="local_demo.meaning", allow_new=True)
    text = "".join(output_text(output) for output in read_nb(path).cells[0].outputs)
    assert "42" in text

    site_root = root / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    site_root.mkdir(parents=True)
    (site_root / "project_only_dep.py").write_text("def answer():\n    return 'project env'\n", encoding="utf-8")
    project_env = nbs / "project_env.ipynb"
    write_nb(str(project_env), "%%code\nfrom project_only_dep import answer\nprint(answer())", replace=True)
    exec_nb(str(project_env), timeout=5, allow="project_only_dep.answer", allow_new=True)
    text = "".join(output_text(output) for output in read_nb(project_env).cells[0].outputs)
    assert "project env" in text
finally:
    remove_demo_path(root)


In [ ]:
root, nbs = _make_execute_project("03_execute_check")
try:
    check = nbs / "check_only.ipynb"
    write_nb(str(check), "%%code\nprint('check only')", replace=True)
    before = check.read_text(encoding="utf-8")
    exec_nb(str(check), timeout=5, allow_new=True, check_only=True)
    assert check.read_text(encoding="utf-8") == before

    cross = nbs / "cross.ipynb"
    write_nb(str(cross), "%%code\nx = 2\n---\n%%code\nprint(x + 3)", replace=True)
    exec_nb(str(cross), timeout=5, allow_new=True)
    text = "".join(output_text(output) for output in read_nb(cross).cells[1].outputs)
    assert "5" in text

    skipped = nbs / "eval_false.ipynb"
    write_nb(str(skipped), "%%code\n#| eval: false\nraise RuntimeError('should not run')", replace=True)
    exec_nb(str(skipped), timeout=5, allow_new=True)
    assert read_nb(skipped).cells[0].outputs == []
finally:
    remove_demo_path(root)


In [ ]:
#| eval: false
root, nbs = _make_execute_project("03_execute_safety")
try:
    slow = nbs / "slow.ipynb"
    write_nb(str(slow), "%%code\nimport time\ntime.sleep(2)", replace=True)
    exec_nb(str(slow), timeout=1, allow_new=True, safe=False)
    cell = read_nb(slow).cells[0]
    assert cell.metadata["nbskill"][_TIMEOUT_HASH_KEY]
    text = "".join(output_text(output) for output in cell.outputs)
    assert "ran longer than 1s" in text
    exec_nb(str(slow), timeout=1, allow_new=True, safe=False)
    cell = read_nb(slow).cells[0]
    text = "".join(output_text(output) for output in cell.outputs)
    assert "skipped cell" in text
    update_cell(str(slow), "print('fast now')", cell_id=cell.id)
    exec_nb(str(slow), timeout=1, allow_new=True, safe=False)
    cell = read_nb(slow).cells[0]
    assert _TIMEOUT_HASH_KEY not in cell.metadata.get("nbskill", {})
    text = "".join(output_text(output) for output in cell.outputs)
    assert "fast now" in text
finally:
    remove_demo_path(root)


In [ ]:
root, nbs = _make_execute_project("03_execute_blocks")
try:
    blocked_cases = {
        "path_write": "from pathlib import Path\nPath('bad.txt').write_text('bad')",
        "open_write": "open('bad.txt', 'w').write('bad')",
        "remove": "import os\nos.remove('missing.txt')",
        "rmtree": "import shutil\nshutil.rmtree('missing')",
        "subprocess": "import subprocess\nsubprocess.run(['true'])",
        "magic": "%time 1 + 1",
        "shell": "!echo unsafe",
    }
    for name, source in blocked_cases.items():
        nb_path = nbs / f"{name}.ipynb"
        write_nb(str(nb_path), f"%%code\n{source}", replace=True)
        exec_nb(str(nb_path), timeout=5, allow_new=True)
        text = "".join(output_text(output) for output in read_nb(nb_path).cells[0].outputs)
        assert text
        if name in {"magic", "shell"}: assert "blocks IPython" in text
finally:
    remove_demo_path(root)


In [ ]:
root, nbs = _make_execute_project("03_execute_write")
try:
    allowed_write = nbs / "allowed_write.ipynb"
    ok_file = root / "allowed.txt"
    write_nb(str(allowed_write), f"%%code\nfrom pathlib import Path\nPath({str(ok_file)!r}).write_text('ok')\nprint(Path({str(ok_file)!r}).read_text())", replace=True)
    exec_nb(str(allowed_write), timeout=5, ok_dests=str(root), allow_new=True)
    assert ok_file.read_text(encoding="utf-8") == "ok"
finally:
    remove_demo_path(root)


In [ ]:
#| eval: false
root, nbs = _make_execute_project("03_execute_httpx")
try:
    import httpx
    live = nbs / "live_httpx.ipynb"
    write_nb(str(live), "%%code\nimport httpx\nhttpx.get('https://api.openai.com/v1/test')", replace=True)
    exec_nb(str(live), timeout=5, allow_new=True)
    text = "".join(output_text(output) for output in read_nb(live).cells[0].outputs)
    assert "blocked live httpx call" in text

    cached = nbs / "cached_httpx.ipynb"
    url = "https://api.openai.com/v1/test"
    request = httpx.Request("GET", url)
    key = _cachy_key(request)
    (root / "cachy.jsonl").write_text(
        json.dumps({"key": key, "response": "cached body", "headers": {"content-type": "text/plain"}, "status_code": 200}) + "\n",
        encoding="utf-8",
    )
    write_nb(str(cached), f"%%code\nimport httpx\nr = httpx.get({url!r})\nprint(r.text)", replace=True)
    exec_nb(str(cached), timeout=5, cache_httpx=True, cache_dir=str(root), allow_new=True)
    text = "".join(output_text(output) for output in read_nb(cached).cells[0].outputs)
    assert "cached body" in text
finally:
    remove_demo_path(root)


In [ ]:
#| eval: false
root, nbs = _make_execute_project("03_execute_httpx_miss")
try:
    miss = nbs / "miss_httpx.ipynb"
    miss_url = "https://api.openai.com/v1/miss"
    write_nb(str(miss), f"%%code\nimport httpx\nhttpx.get({miss_url!r})", replace=True)
    exec_nb(str(miss), timeout=5, cache_httpx=True, cache_dir=str(root), allow_new=True)
    text = "".join(output_text(output) for output in read_nb(miss).cells[0].outputs)
    assert "no cached httpx response" in text
finally:
    remove_demo_path(root)


In [ ]:
warning_root = demo_path("03_execute_uncalled_warning")
try:
    warning_root.mkdir()
    warning_nbs = warning_root / "nbs"
    warning_nbs.mkdir()
    uncalled = warning_nbs / "uncalled.ipynb"
    write_nb(str(uncalled), "%%code\ndef unused_demo():\n    return 1\n---\n%%code\nprint('done')", replace=True)
    out = StringIO()
    with redirect_stdout(out):
        exec_nb(str(uncalled), timeout=5, allow_new=True, show_output=False)
    warning_text = out.getvalue()
    assert "Execution warnings:" in warning_text
    assert "unused_demo" in warning_text
finally:
    remove_demo_path(warning_root)


In [ ]:
warning_root = demo_path("03_execute_called_warning")
try:
    warning_root.mkdir()
    warning_nbs = warning_root / "nbs"
    warning_nbs.mkdir()
    called = warning_nbs / "called.ipynb"
    write_nb(str(called), "%%code\ndef used_demo():\n    return 1\n---\n%%code\nused_demo()", replace=True)
    out = StringIO()
    with redirect_stdout(out):
        exec_nb(str(called), timeout=5, allow_new=True, show_output=False)
    assert "used_demo" not in out.getvalue()
finally:
    remove_demo_path(warning_root)


In [ ]:
warning_root = demo_path("03_execute_decorated_warning")
try:
    warning_root.mkdir()
    warning_nbs = warning_root / "nbs"
    warning_nbs.mkdir()
    decorated = warning_nbs / "decorated.ipynb"
    write_nb(str(decorated), "%%code\ndef _route(func):\n    return func\n\n@_route\ndef api_handler():\n    return 1", replace=True)
    out = StringIO()
    with redirect_stdout(out):
        exec_nb(str(decorated), timeout=5, allow_new=True, show_output=False)
    assert "api_handler" not in out.getvalue()
finally:
    remove_demo_path(warning_root)


### Testing after edits

The write tools can ask for a notebook test immediately after changing a notebook. These helpers summarize saved error outputs so a failed write/test cycle reports the useful cell-level problem.

In [ ]:
#| export
def _notebook_error_summaries(path, up2id=None):
    with notebook_locks(path):
        nb = read_nb(path)
        errors = []
        for idx, cell in _executed_cells(nb, up2id=up2id):
            for output in cell.get("outputs", []):
                if output.get("output_type") == "error":
                    ename = output.get("ename", "Error")
                    evalue = output.get("evalue", "")
                    errors.append(f"id={cell.id} {ename}: {evalue}".strip())
        return errors

In [ ]:
#| export
def run_notebook_test(path, timeout=30):
    print(f"Running notebook test with safe execution on {path} (timeout={timeout}s)")
    _execute_nb(path, dest=path, exc_stop=False, timeout=timeout, verbose=False)
    _print_nb_outputs(path)
    errors = _notebook_error_summaries(path)
    if errors:
        sys.stdout.flush()
        cli_error("Notebook test failed after writing/execution: " + "; ".join(errors))

In [ ]:
#| hide
root, nbs = _make_execute_project("03_execute_policy")
try:
    pure = nbs / "pure.ipynb"
    write_nb(str(pure), "%%code\ndef helper():\n    return 11\n---\n%%code\nprint(helper())", replace=True)
    exec_nb(str(pure), timeout=5)
    text = "".join(output_text(output) for cell in read_nb(pure).cells for output in cell.outputs)
    policy = json.loads(_execution_policy_path(pure).read_text(encoding="utf-8"))
    assert "11" in text
    assert policy["functions"]["helper"]["allowed"] is True

    unsafe = nbs / "unsafe.ipynb"
    out_file = root / "made.txt"
    write_nb(
        str(unsafe),
        f"%%code\nfrom pathlib import Path\n\ndef write_file():\n    Path({str(out_file)!r}).write_text('ok')\n---\n%%code\nwrite_file()",
        replace=True,
    )
    exec_nb(str(unsafe), timeout=5, ok_dests=str(root))
    text = "".join(output_text(output) for cell in read_nb(unsafe).cells for output in cell.outputs)
    policy_path = _execution_policy_path(unsafe)
    policy = json.loads(policy_path.read_text(encoding="utf-8"))
    assert policy["functions"]["write_file"]["allowed"] is False
    assert "allowed=false" in text
    assert not out_file.exists()

    policy["functions"]["write_file"]["allowed"] = True
    policy_path.write_text(json.dumps(policy, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    exec_nb(str(unsafe), timeout=5, ok_dests=str(root))
    assert out_file.read_text(encoding="utf-8") == "ok"
finally:
    remove_demo_path(root)

In [ ]:
#| hide
root, nbs = _make_execute_project("03_execute_policy_stale")
try:
    stale = nbs / "stale.ipynb"
    write_nb(str(stale), "%%code\ndef removed():\n    return 1", replace=True)
    exec_nb(str(stale), timeout=5)
    assert "removed" in json.loads(_execution_policy_path(stale).read_text(encoding="utf-8"))["functions"]

    write_nb(str(stale), "%%code\ndef current():\n    return 2", replace=True)
    exec_nb(str(stale), timeout=5)
    functions = json.loads(_execution_policy_path(stale).read_text(encoding="utf-8"))["functions"]
    assert "current" in functions and "removed" not in functions
finally: remove_demo_path(root)

In [ ]:
#| hide
root, nbs = _make_execute_project("03_execute_policy_source_change")
try:
    path = nbs / "source_change.ipynb"
    write_nb(str(path), "%%code\ndef helper():\n    return 1", replace=True)
    policy = _sync_execution_policy(path, read_nb(path))
    assert policy["functions"]["helper"]["allowed"] is True

    nb = read_nb(path)
    nb.cells[0].source = "from pathlib import Path\n\ndef helper():\n    Path('out.txt').write_text('x')"
    policy = _sync_execution_policy(path, nb)
    assert policy["functions"]["helper"]["allowed"] is False
finally: remove_demo_path(root)

In [ ]:
#| hide
root, nbs = _make_execute_project("03_execute_policy_database")
try:
    path = nbs / "database.ipynb"
    write_nb(str(path), "import sqlite3\n\ndef connect():\n    return sqlite3.connect('live.db')", replace=True)
    policy = _sync_execution_policy(path, read_nb(path))
    entry = policy["functions"]["connect"]
    assert entry["allowed"] is False
    assert "call:sqlite3.connect" in entry["effects"]
finally: remove_demo_path(root)